# Unified — cargo check on Google Colab

Runs `cargo check` for the private repo **ArrayIterator/unified** on Colab's free runtime (~12 GB RAM, 2 vCPU), which is enough for this codebase — the 6 GB VM it was checked on before ran out of memory (rustc needs ~5.3 GB).

**You need:** a GitHub Personal Access Token (classic) with the `repo` scope. Create one at https://github.com/settings/tokens — it is typed into a password prompt below, never printed, and never written into this notebook.

Run the cells top to bottom. The Colab session is ephemeral: the clone and the build cache disappear when the runtime disconnects.

In [ ]:
# 1. Clone the private repo (token asked via password prompt, stored only in ~/.git-credentials with 0600 perms)
import getpass, os

token = getpass.getpass('GitHub PAT (repo scope): ')
cred_path = os.path.expanduser('~/.git-credentials')
with open(cred_path, 'w') as f:
    f.write(f'https://{token}@github.com\n')
os.chmod(cred_path, 0o600)
del token

!git config --global credential.helper store
!rm -rf /content/unified && git clone https://github.com/ArrayIterator/unified.git /content/unified
!cd /content/unified && git log --oneline -3 && git status --short | head -20

In [ ]:
# 2. Install Rust (stable) if cargo is missing, then make it available to later cells
!command -v cargo >/dev/null || (curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable --profile minimal)

import os
cargo_bin = os.path.expanduser('~/.cargo/bin')
if cargo_bin not in os.environ['PATH']:
    os.environ['PATH'] = cargo_bin + ':' + os.environ['PATH']
os.environ['CARGO_BUILD_JOBS'] = '2'  # Colab free tier: 2 vCPU

!rustc --version && cargo --version

In [ ]:
# 3. Known dependency fixes (already verified against crates.io; applied only if the broken lines are still present)
#    - x509-certificate 0.27 does not exist on crates.io (latest is 0.25.0) -> pin 0.25
#    - reqwest 0.12 has no feature named 'rustls' -> use 'rustls-tls-webpki-roots'
!cd /content/unified && sed -i 's/^x509-certificate = "0.27"/x509-certificate = "0.25"/' Cargo.toml
!cd /content/unified && sed -i 's/"rustls"/"rustls-tls-webpki-roots"/' Cargo.toml
!cd /content/unified && git diff -- Cargo.toml | head -40

In [ ]:
# 4. cargo check (this takes a while on first run: full dependency download + build)
!cd /content/unified && cargo check 2>&1 | tee /content/unified/cargo-check.log; echo "EXIT: ${PIPESTATUS[0]}"

In [ ]:
# 5. Summary: error count + first errors (if any)
!echo "--- error lines: $(grep -c '^error' /content/unified/cargo-check.log || true)" && grep -m 20 '^error' /content/unified/cargo-check.log || echo 'No error lines found.'
!tail -30 /content/unified/cargo-check.log

## Notes

- If the check passes here, the code is sound — the earlier failures were the VM running out of RAM, not code errors.
- `cargo run` smoke test is intentionally not in this notebook: it needs a PostgreSQL database and a config file. Run it only after `cargo check` passes, with `unified -t` first to validate the config.
- The log is saved at `/content/unified/cargo-check.log`. Download it from the Colab file browser if you want to keep it — it is gone when the runtime disconnects.
- If the repo on GitHub does not yet contain the latest refactor (the push happens once everything is 100% ready), re-run this notebook after the push.